In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement the k-means clustering algorithm for 2D points. Given arrays of x and y coordinates for data points, initial centroids, and other parameters, assign each point to the nearest centroid and update the centroids iteratively. The final centroids and labels should be stored in the output arrays.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in <code>labels</code>, <code>final_centroid_x</code>, and <code>final_centroid_y</code></li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:
sample_size = 4, k = 2, max_iterations = 10
data_x = [1.0, 2.0, 8.0, 9.0]
data_y = [1.0, 2.0, 8.0, 9.0]
initial_centroid_x = [1.0, 8.0]
initial_centroid_y = [1.0, 8.0]
Output: (see reference implementation for expected output)
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ sample_size ≤ 1000000</li>
  <li>1 ≤ k ≤ 1000</li>
  <li>All arrays are float32 except labels, which is int32</li>

  <li>Performance is measured with <code>k</code> = 5, <code>max_iterations</code> = 30, <code>sample_size</code> = 10,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// data_x, data_y, labels, initial_centroid_x, initial_centroid_y,
// final_centroid_x, final_centroid_y are device pointers
extern "C" void solve(const float* data_x, const float* data_y, int* labels,
                      float* initial_centroid_x, float* initial_centroid_y, float* final_centroid_x,
                      float* final_centroid_y, int sample_size, int k, int max_iterations) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# data_x, data_y, labels, initial_centroid_x, initial_centroid_y,
# final_centroid_x, final_centroid_y are tensors on the GPU
@cute.jit
def solve(
    data_x: cute.Tensor,
    data_y: cute.Tensor,
    labels: cute.Tensor,
    initial_centroid_x: cute.Tensor,
    initial_centroid_y: cute.Tensor,
    final_centroid_x: cute.Tensor,
    final_centroid_y: cute.Tensor,
    sample_size: cute.Int32,
    k: cute.Int32,
    max_iterations: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# data_x, data_y, initial_centroid_x, initial_centroid_y are tensors on the GPU
@jax.jit
def solve(
    data_x: jax.Array,
    data_y: jax.Array,
    initial_centroid_x: jax.Array,
    initial_centroid_y: jax.Array,
    sample_size: int,
    k: int,
    max_iterations: int,
) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    data_x: UnsafePointer[Float32, MutExternalOrigin],
    data_y: UnsafePointer[Float32, MutExternalOrigin],
    labels: UnsafePointer[Int32, MutExternalOrigin],
    initial_centroid_x: UnsafePointer[Float32, MutExternalOrigin],
    initial_centroid_y: UnsafePointer[Float32, MutExternalOrigin],
    final_centroid_x: UnsafePointer[Float32, MutExternalOrigin],
    final_centroid_y: UnsafePointer[Float32, MutExternalOrigin],
    sample_size: Int32,
    k: Int32,
    max_iterations: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# data_x, data_y, labels, initial_centroid_x,
# initial_centroid_y, final_centroid_x, final_centroid_y are tensors on the GPU
def solve(
    data_x: torch.Tensor,
    data_y: torch.Tensor,
    labels: torch.Tensor,
    initial_centroid_x: torch.Tensor,
    initial_centroid_y: torch.Tensor,
    final_centroid_x: torch.Tensor,
    final_centroid_y: torch.Tensor,
    sample_size: int,
    k: int,
    max_iterations: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# data_x, data_y, labels, initial_centroid_x,
# initial_centroid_y, final_centroid_x, final_centroid_y are tensors on the GPU
def solve(
    data_x: torch.Tensor,
    data_y: torch.Tensor,
    labels: torch.Tensor,
    initial_centroid_x: torch.Tensor,
    initial_centroid_y: torch.Tensor,
    final_centroid_x: torch.Tensor,
    final_centroid_y: torch.Tensor,
    sample_size: int,
    k: int,
    max_iterations: int,
):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="K-Means Clustering", atol=1e-04, rtol=1e-04, num_gpus=1, access_tier="free"
        )

    def reference_impl(
        self,
        data_x: torch.Tensor,
        data_y: torch.Tensor,
        labels: torch.Tensor,
        initial_centroid_x: torch.Tensor,
        initial_centroid_y: torch.Tensor,
        final_centroid_x: torch.Tensor,
        final_centroid_y: torch.Tensor,
        sample_size: int,
        k: int,
        max_iterations: int,
    ):
        assert data_x.shape == (sample_size,)
        assert data_y.shape == (sample_size,)
        assert initial_centroid_x.shape == (k,)
        assert initial_centroid_y.shape == (k,)
        assert final_centroid_x.shape == (k,)
        assert final_centroid_y.shape == (k,)
        assert labels.shape == (sample_size,)
        final_centroid_x.copy_(initial_centroid_x)
        final_centroid_y.copy_(initial_centroid_y)
        for _ in range(max_iterations):
            expanded_x = data_x.view(-1, 1) - final_centroid_x.view(1, -1)
            expanded_y = data_y.view(-1, 1) - final_centroid_y.view(1, -1)
            distances = expanded_x**2 + expanded_y**2
            labels.copy_(torch.argmin(distances, dim=1))
            for i in range(k):
                mask = labels == i
                if mask.any():
                    final_centroid_x[i] = data_x[mask].mean()
                    final_centroid_y[i] = data_y[mask].mean()

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "data_x": (ctypes.POINTER(ctypes.c_float), "in"),
            "data_y": (ctypes.POINTER(ctypes.c_float), "in"),
            "labels": (ctypes.POINTER(ctypes.c_int), "out"),
            "initial_centroid_x": (ctypes.POINTER(ctypes.c_float), "in"),
            "initial_centroid_y": (ctypes.POINTER(ctypes.c_float), "in"),
            "final_centroid_x": (ctypes.POINTER(ctypes.c_float), "out"),
            "final_centroid_y": (ctypes.POINTER(ctypes.c_float), "out"),
            "sample_size": (ctypes.c_int, "in"),
            "k": (ctypes.c_int, "in"),
            "max_iterations": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        sample_size, k, max_iterations = 4, 2, 10
        data_x = torch.tensor([1.0, 2.0, 8.0, 9.0], device="cuda", dtype=dtype)
        data_y = torch.tensor([1.0, 2.0, 8.0, 9.0], device="cuda", dtype=dtype)
        labels = torch.empty(sample_size, device="cuda", dtype=torch.int32)
        initial_centroid_x = torch.tensor([1.0, 8.0], device="cuda", dtype=dtype)
        initial_centroid_y = torch.tensor([1.0, 8.0], device="cuda", dtype=dtype)
        final_centroid_x = torch.empty(k, device="cuda", dtype=dtype)
        final_centroid_y = torch.empty(k, device="cuda", dtype=dtype)
        return {
            "data_x": data_x,
            "data_y": data_y,
            "labels": labels,
            "initial_centroid_x": initial_centroid_x,
            "initial_centroid_y": initial_centroid_y,
            "final_centroid_x": final_centroid_x,
            "final_centroid_y": final_centroid_y,
            "sample_size": sample_size,
            "k": k,
            "max_iterations": max_iterations,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        test_cases = []
        # basic_clustering
        data_x = torch.tensor(
            [1.0, 1.5, 1.2, 1.3, 1.1, 5.0, 5.2, 5.1, 5.3, 5.4, 10.1, 10.2, 10.0, 10.3, 10.5],
            device="cuda",
            dtype=dtype,
        )
        data_y = torch.tensor(
            [1.0, 1.5, 1.2, 1.3, 1.1, 5.0, 5.2, 5.1, 5.3, 5.4, 10.1, 10.2, 10.0, 10.3, 10.5],
            device="cuda",
            dtype=dtype,
        )
        labels = torch.empty(15, device="cuda", dtype=torch.int32)
        initial_centroid_x = torch.tensor([3.4, 7.1, 8.5], device="cuda", dtype=dtype)
        initial_centroid_y = torch.tensor([3.4, 7.1, 8.5], device="cuda", dtype=dtype)
        final_centroid_x = torch.empty(3, device="cuda", dtype=dtype)
        final_centroid_y = torch.empty(3, device="cuda", dtype=dtype)
        test_cases.append(
            {
                "data_x": data_x,
                "data_y": data_y,
                "labels": labels,
                "initial_centroid_x": initial_centroid_x,
                "initial_centroid_y": initial_centroid_y,
                "final_centroid_x": final_centroid_x,
                "final_centroid_y": final_centroid_y,
                "sample_size": 15,
                "k": 3,
                "max_iterations": 20,
            }
        )
        # single_cluster
        data_x = torch.tensor(
            [1.0, 1.2, 1.1, 1.3, 1.5, 1.4, 1.6, 1.2, 1.3, 1.1], device="cuda", dtype=dtype
        )
        data_y = torch.tensor(
            [1.0, 1.2, 1.1, 1.3, 1.5, 1.4, 1.6, 1.2, 1.3, 1.1], device="cuda", dtype=dtype
        )
        labels = torch.empty(10, device="cuda", dtype=torch.int32)
        initial_centroid_x = torch.tensor([1.0, 5.0, 10.0], device="cuda", dtype=dtype)
        initial_centroid_y = torch.tensor([1.0, 5.0, 10.0], device="cuda", dtype=dtype)
        final_centroid_x = torch.empty(3, device="cuda", dtype=dtype)
        final_centroid_y = torch.empty(3, device="cuda", dtype=dtype)
        test_cases.append(
            {
                "data_x": data_x,
                "data_y": data_y,
                "labels": labels,
                "initial_centroid_x": initial_centroid_x,
                "initial_centroid_y": initial_centroid_y,
                "final_centroid_x": final_centroid_x,
                "final_centroid_y": final_centroid_y,
                "sample_size": 10,
                "k": 3,
                "max_iterations": 10,
            }
        )
        # empty_clusters
        data_x = torch.tensor(
            [
                1.0,
                1.5,
                1.2,
                1.3,
                1.1,
                1.4,
                1.6,
                1.2,
                1.7,
                1.3,
                10.0,
                10.5,
                10.2,
                10.3,
                10.1,
                10.4,
                10.6,
                10.2,
                10.7,
                10.3,
            ],
            device="cuda",
            dtype=dtype,
        )
        data_y = torch.tensor(
            [
                1.0,
                1.5,
                1.2,
                1.3,
                1.1,
                1.4,
                1.6,
                1.2,
                1.7,
                1.3,
                10.0,
                10.5,
                10.2,
                10.3,
                10.1,
                10.4,
                10.6,
                10.2,
                10.7,
                10.3,
            ],
            device="cuda",
            dtype=dtype,
        )
        labels = torch.empty(20, device="cuda", dtype=torch.int32)
        initial_centroid_x = torch.tensor([1.5, 5.0, 10.5], device="cuda", dtype=dtype)
        initial_centroid_y = torch.tensor([1.5, 5.0, 10.5], device="cuda", dtype=dtype)
        final_centroid_x = torch.empty(3, device="cuda", dtype=dtype)
        final_centroid_y = torch.empty(3, device="cuda", dtype=dtype)
        test_cases.append(
            {
                "data_x": data_x,
                "data_y": data_y,
                "labels": labels,
                "initial_centroid_x": initial_centroid_x,
                "initial_centroid_y": initial_centroid_y,
                "final_centroid_x": final_centroid_x,
                "final_centroid_y": final_centroid_y,
                "sample_size": 20,
                "k": 3,
                "max_iterations": 15,
            }
        )
        # max_iterations_limit
        data_x = torch.tensor(
            [1.0, 1.5, 1.2, 1.3, 1.1, 5.0, 5.2, 5.1, 5.3, 5.4, 10.1, 10.2, 10.0, 10.3, 10.5],
            device="cuda",
            dtype=dtype,
        )
        data_y = torch.tensor(
            [1.0, 1.5, 1.2, 1.3, 1.1, 5.0, 5.2, 5.1, 5.3, 5.4, 10.1, 10.2, 10.0, 10.3, 10.5],
            device="cuda",
            dtype=dtype,
        )
        labels = torch.empty(15, device="cuda", dtype=torch.int32)
        initial_centroid_x = torch.tensor([3.4, 7.1, 8.5], device="cuda", dtype=dtype)
        initial_centroid_y = torch.tensor([3.4, 7.1, 8.5], device="cuda", dtype=dtype)
        final_centroid_x = torch.empty(3, device="cuda", dtype=dtype)
        final_centroid_y = torch.empty(3, device="cuda", dtype=dtype)
        test_cases.append(
            {
                "data_x": data_x,
                "data_y": data_y,
                "labels": labels,
                "initial_centroid_x": initial_centroid_x,
                "initial_centroid_y": initial_centroid_y,
                "final_centroid_x": final_centroid_x,
                "final_centroid_y": final_centroid_y,
                "sample_size": 15,
                "k": 3,
                "max_iterations": 5,
            }
        )
        # medium_random
        sample_size = 100
        k = 5
        data_x = torch.empty(sample_size, device="cuda", dtype=dtype).uniform_(0.0, 100.0)
        data_y = torch.empty(sample_size, device="cuda", dtype=dtype).uniform_(0.0, 100.0)
        labels = torch.empty(sample_size, device="cuda", dtype=torch.int32)
        initial_centroid_x = torch.tensor(
            [20.0, 40.0, 60.0, 80.0, 10.0], device="cuda", dtype=dtype
        )
        initial_centroid_y = torch.tensor(
            [20.0, 40.0, 60.0, 80.0, 50.0], device="cuda", dtype=dtype
        )
        final_centroid_x = torch.empty(k, device="cuda", dtype=dtype)
        final_centroid_y = torch.empty(k, device="cuda", dtype=dtype)
        test_cases.append(
            {
                "data_x": data_x,
                "data_y": data_y,
                "labels": labels,
                "initial_centroid_x": initial_centroid_x,
                "initial_centroid_y": initial_centroid_y,
                "final_centroid_x": final_centroid_x,
                "final_centroid_y": final_centroid_y,
                "sample_size": sample_size,
                "k": k,
                "max_iterations": 30,
            }
        )
        return test_cases

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        sample_size = 10000
        k = 5
        data_x = torch.empty(sample_size, device="cuda", dtype=dtype).uniform_(0.0, 1000.0)
        data_y = torch.empty(sample_size, device="cuda", dtype=dtype).uniform_(0.0, 1000.0)
        labels = torch.empty(sample_size, device="cuda", dtype=torch.int32)
        initial_centroid_x = torch.tensor(
            [100.0, 200.0, 300.0, 400.0, 500.0], device="cuda", dtype=dtype
        )
        initial_centroid_y = torch.tensor(
            [100.0, 200.0, 300.0, 400.0, 500.0], device="cuda", dtype=dtype
        )
        final_centroid_x = torch.empty(k, device="cuda", dtype=dtype)
        final_centroid_y = torch.empty(k, device="cuda", dtype=dtype)
        return {
            "data_x": data_x,
            "data_y": data_y,
            "labels": labels,
            "initial_centroid_x": initial_centroid_x,
            "initial_centroid_y": initial_centroid_y,
            "final_centroid_x": final_centroid_x,
            "final_centroid_y": final_centroid_y,
            "sample_size": sample_size,
            "k": k,
            "max_iterations": 30,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
